# Web Scraping Workshop: Local HTML Scraping with BeautifulSoup

Welcome to the Web Scraping Workshop! In this hands-on lab, we will learn how to extract structured data from HTML web pages using Python and **BeautifulSoup**.

## Learning Objectives

By the end of this workshop, you will be able to:
- Read local HTML files into Python using `with open(...)`
- Parse raw HTML strings into a searchable document tree using `BeautifulSoup`
- Locate unique elements using HTML tags and the `id` attribute
- Locate collections of elements using CSS class names (`class_`)
- Apply the **Container Pattern** to extract multi-field records (title, author, price) reliably
- Clean raw text strings, strip currency symbols, and compute summary statistics

---

### The Web Scraping Mental Model

Before writing code, let's understand what HTML is. An HTML document is a structured tree of nested tags:

```text
┌─────────────────────────────────────────────────────────────┐
│ <html>                                                      │
│   ├── <head>                                                │
│   │     └── <title>Fake Bookstore</title>                   │
│   └── <body>                                                │
│         ├── <h1 id="site-title">Welcome...</h1>             │
│         └── <div class="book-container">                    │
│               ├── <div class="book"> ... </div>  (Card 1)   │
│               ├── <div class="book"> ... </div>  (Card 2)   │
│               └── <div class="book"> ... </div>  (Card 3)   │
└─────────────────────────────────────────────────────────────┘
```

**Goal:** Turn messy HTML into clean, structured Python dictionaries and summary statistics.

In [1]:
# Setup: Environment check, imports, and sandbox data verification
import sys
import os
from bs4 import BeautifulSoup

# 1. Environment check
print(f"Python version: {sys.version.split()[0]} | Platform: {sys.platform}")

# 2. Zero-404 sandbox data check: Ensure fake_bookstore.html exists locally
HTML_SAMPLE = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Fake Bookstore</title>
</head>
<body>
    <h1 id="site-title">Welcome to the Fake Bookstore</h1>
    <p>This is a simple page to practice web scraping. No CSS, just pure HTML structure.</p>

    <div class="book-container">
        <div class="book">
            <h2 class="book-title">The Python Learner</h2>
            <p class="author">By Jane Doe</p>
            <p class="price">$29.99</p>
        </div>

        <div class="book">
            <h2 class="book-title">Data Science 101</h2>
            <p class="author">By John Smith</p>
            <p class="price">$35.50</p>
        </div>

        <div class="book">
            <h2 class="book-title">Web Scraping for Beginners</h2>
            <p class="author">By Alice Johnson</p>
            <p class="price">$19.99</p>
        </div>
    </div>
</body>
</html>
'''

if not os.path.exists("fake_bookstore.html"):
    with open("fake_bookstore.html", "w", encoding="utf-8") as f:
        f.write(HTML_SAMPLE.strip())
    print("📁 Created sandbox file: fake_bookstore.html")
else:
    print("📁 Found existing sandbox file: fake_bookstore.html")

print("✅ Setup complete: BeautifulSoup is ready and sandbox files are verified!")

Python version: 3.12.10 | Platform: win32
📁 Found existing sandbox file: fake_bookstore.html
✅ Setup complete: BeautifulSoup is ready and sandbox files are verified!


## Part 1 — Guided Mechanics: Parsing & Basic Search

### 1.1 Reading the Local HTML File

Before BeautifulSoup can search an HTML page, we must read the file from disk into Python memory as a standard text string.

```text
with open("fake_bookstore.html", "r", encoding="utf-8") as file:
 │    │                           │    └─ handles symbols and accents cleanly
 │    │                           └─ "r" stands for read-only mode
 │    └─ file path on your computer
 └─ context manager that automatically closes the file when done

    html_content = file.read()  ← reads entire file into a Python str
```

- `open(...)` opens the file.
- The `with` statement ensures the file is closed cleanly, even if an error occurs.
- `.read()` returns the complete contents as a Python string (`str`).

**Expected output:** `Loaded fake_bookstore.html (934 characters)` and the first lines of HTML printed.

In [2]:
# 1.1 — Solution: Read fake_bookstore.html into a string variable named html_content
with open("fake_bookstore.html", "r", encoding="utf-8") as file:
    html_content = file.read()

print(f"Loaded fake_bookstore.html ({len(html_content)} characters)")
print("First 150 characters:")
print(html_content[:150])

# Self-check assertion
assert isinstance(html_content, str) and len(html_content) > 0, "html_content must be a non-empty string!"
assert "Fake Bookstore" in html_content, "Missing site title in loaded content!"
print("✅ Test passed!")

Loaded fake_bookstore.html (904 characters)
First 150 characters:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Fake Bookstore</title>
</head>
<body>
    <h1 id="site-title">Welcome to
✅ Test passed!


### 1.2 Creating the BeautifulSoup Object

A raw HTML string is just plain text to Python. We cannot ask `html_content` to "give me the headers" without complex text search.

**BeautifulSoup** parses the raw text and constructs an object tree that mimics the structure of the web page.

```text
soup = BeautifulSoup(html_content, "html.parser")
│      │             │             └─ the parsing engine (built into Python)
│      │             └─ your raw HTML string
│      └─ the BeautifulSoup class imported from bs4
└─ the parsed tree object ready for searching
```

You can directly access top-level tags like `soup.title` to inspect the page header:
- `soup.title` returns the entire `<title>Fake Bookstore</title>` tag object.
- `soup.title.text.strip()` extracts just the plain text string `'Fake Bookstore'`.

**Expected output:** `Page title: Fake Bookstore`

In [3]:
# 1.2 — Solution: Parse the HTML string with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

# Extract the <title> tag text
page_title = soup.title.text.strip()
print("Page title:", page_title)

# Self-check assertion
assert page_title == "Fake Bookstore", f"Expected 'Fake Bookstore', got '{page_title}'"
print("✅ Test passed!")

Page title: Fake Bookstore
✅ Test passed!


### 1.3 Finding a Unique Element by ID (`.find()`)

In HTML, an `id` is a unique identifier: in valid HTML, no two elements on the same page share the same `id`.

To locate an element with a specific ID, use `.find()` with the `id` parameter:

```text
header_tag = soup.find("h1", id="site-title")
             │         │     └─ keyword argument targeting id="site-title"
             │         └─ tag name to search for
             └─ searches the tree and returns the FIRST matching Tag, or None
```

Once you have a Tag:
- `header_tag.text` returns the string inside the tag.
- `.strip()` removes any surrounding newline characters (`\n`) or spaces.

**Expected output:** `Header: Welcome to the Fake Bookstore`

In [4]:
# 1.3 — Solution: Find the main header by its unique ID
header_tag = soup.find("h1", id="site-title")

# Extract the text and strip whitespace
header_text = header_tag.text.strip()
print("Header:", header_text)

# Self-check assertion
assert header_text == "Welcome to the Fake Bookstore", f"Unexpected header: {header_text}"
print("✅ Test passed!")

Header: Welcome to the Fake Bookstore
✅ Test passed!


### 1.4 Finding Elements by Class Name (`.find_all()`)

While an `id` is unique, CSS **classes** are shared across multiple elements to apply consistent styling.

To find all elements sharing a class, use `.find_all()`:

```text
title_tags = soup.find_all("h2", class_="book-title")
             │             │     │
             │             │     └─ IMPORTANT: Notice the trailing underscore 'class_'!
             │             │        In Python, 'class' is a reserved keyword for OOP.
             │             │        BeautifulSoup uses 'class_' to avoid a syntax conflict.
             │             └─ tag name to match
             └─ returns a Python list of ALL matching Tag objects
```

Key difference between methods:
- `.find()` → Returns **one** Tag object (the first match), or `None`.
- `.find_all()` → Returns a Python **list** of Tag objects (empty `[]` if no matches).

**Expected output:** Found 3 titles: `['The Python Learner', 'Data Science 101', 'Web Scraping for Beginners']`

In [5]:
# 1.4 — Solution: Extract all book titles using class_="book-title"
title_tags = soup.find_all("h2", class_="book-title")

# Extract the text from each tag in the list
all_titles = [tag.text.strip() for tag in title_tags]
print(f"Found {len(all_titles)} titles:")
for title in all_titles:
    print(" -", title)

# Self-check assertion
expected = ["The Python Learner", "Data Science 101", "Web Scraping for Beginners"]
assert all_titles == expected, f"Expected {expected}, got {all_titles}"
print("✅ Test passed!")

Found 3 titles:
 - The Python Learner
 - Data Science 101
 - Web Scraping for Beginners
✅ Test passed!


## Part 2 — Guided Application: Container Cards & Calculations

### 2.1 The Container Pattern (Scraping Cards)

A common mistake made by beginners is to run separate searches for titles, authors, and prices:
- `titles = soup.find_all("h2", class_="book-title")`
- `authors = soup.find_all("p", class_="author")`
- `prices = soup.find_all("p", class_="price")`

⚠️ **Why this is risky:** If just one book on a page is missing an author or a price, the lists become different lengths! Index 2 of `prices` will end up attached to the wrong book!

**The Professional Solution: The Container Pattern**
Find the parent card container first, then extract each field from *inside* that specific card:

```text
<div class="book">             ← Parent Container (Card)
 ├── <h2 class="book-title">   ← Title inside this card
 ├── <p class="author">        ← Author inside this card
 └── <p class="price">         ← Price inside this card
```

Let's start by finding all parent cards with class `"book"`.

**Expected output:** `Found 3 book cards.`

In [6]:
# 2.1 — Solution: Locate all parent book cards (<div class="book">)
book_cards = soup.find_all("div", class_="book")

print(f"Found {len(book_cards)} book cards.")

# Self-check assertion
assert len(book_cards) == 3, f"Expected 3 book cards, got {len(book_cards)}"
print("✅ Test passed!")

Found 3 book cards.
✅ Test passed!


### 2.2 Extracting Multi-Field Records per Card

Now, loop through each card in `book_cards`. For each card, use `.find()` to locate its title, author, and price, and bundle them into a Python dictionary.

```text
for card in book_cards:
    title  = card.find("h2", class_="book-title").text.strip()
    author = card.find("p", class_="author").text.strip()
    price  = card.find("p", class_="price").text.strip()
    
    # Store into structured dictionary
    books_catalog.append({
        "title": title,
        "author": author,
        "price_raw": price
    })
```

Because `.find()` is called on `card` (not on `soup`), it searches *only inside that single card*. If a book has missing data, we can detect it immediately without corrupting other records.

**Expected output:** 3 book records printed, each containing title, author, and raw price.

In [7]:
# 2.2 — Solution: Loop through each card and extract all three fields
books_catalog = []

for card in book_cards:
    title = card.find("h2", class_="book-title").text.strip()
    author = card.find("p", class_="author").text.strip()
    price_raw = card.find("p", class_="price").text.strip()
    
    books_catalog.append({
        "title": title,
        "author": author,
        "price_raw": price_raw
    })

for item in books_catalog:
    print(f"📖 {item['title']} | ✍️ {item['author']} | 💵 {item['price_raw']}")

# Self-check assertions
assert len(books_catalog) == 3, f"Expected 3 items, got {len(books_catalog)}"
assert books_catalog[0]["author"] == "By Jane Doe", f"Unexpected author: {books_catalog[0]['author']}"
assert books_catalog[2]["price_raw"] == "$19.99", f"Unexpected price: {books_catalog[2]['price_raw']}"
print("✅ Test passed!")

📖 The Python Learner | ✍️ By Jane Doe | 💵 $29.99
📖 Data Science 101 | ✍️ By John Smith | 💵 $35.50
📖 Web Scraping for Beginners | ✍️ By Alice Johnson | 💵 $19.99
✅ Test passed!


### 2.3 Data Cleaning & Summary Statistics

Web scraping extracts strings, but data analysis requires numbers. Currently, our prices look like `"$29.99"`.

To calculate the total inventory cost and average price, we must clean each price string:

```text
clean_price = float(raw_price.replace("$", "").strip())
              │     │         │
              │     │         └─ replaces '$' with an empty string: "$29.99" -> "29.99"
              │     └─ removes extra whitespace
              └─ converts the cleaned string "29.99" into float 29.99
```

Let's clean the prices and compute:
1. `total_inventory_cost`: the sum of all book prices
2. `average_book_price`: the average price rounded to 2 decimal places

**Expected output:** `Total: $85.48`, `Average: $28.49`

In [8]:
# 2.3 — Solution: Clean prices and calculate summary statistics
prices = []

for item in books_catalog:
    clean_price = float(item["price_raw"].replace("$", "").strip())
    item["price"] = clean_price
    prices.append(clean_price)

total_inventory_cost = round(sum(prices), 2)
average_book_price = round(total_inventory_cost / len(prices), 2)

print(f"Cleaned prices: {prices}")
print(f"Total inventory cost: ${total_inventory_cost:.2f}")
print(f"Average book price:   ${average_book_price:.2f}")

# Self-check assertions
assert total_inventory_cost == 85.48, f"Expected 85.48, got {total_inventory_cost}"
assert average_book_price == 28.49, f"Expected 28.49, got {average_book_price}"
print("✅ Test passed!")

Cleaned prices: [29.99, 35.5, 19.99]
Total inventory cost: $85.48
Average book price:   $28.49
✅ Test passed!


## Bug Hunt — read the traceback like a detective

The cell below contains the single most common bug encountered by web scrapers: `AttributeError: 'NoneType' object has no attribute 'text'`.

Run the cell first and let it fail. Notice the traceback:

```text
Traceback (most recent call last):
  File "<ipython-input>", line 4, in <module>
    print(header.text)
AttributeError: 'NoneType' object has no attribute 'text'
│              └─ Python says: header is None! None does not have a .text attribute!
└─ the error type
```

### Why did this happen?
When `soup.find(...)` cannot find a match on the page, it does not crash — it returns `None`. The crash happens on the *next* line when you try to call `.text` on `None`!

Inspect the `id` name in the code below against the actual HTML in `fake_bookstore.html`, fix the bug, and re-run the cell.

**Goal:** Fix the selector so the header text prints cleanly without crashing.

In [9]:
# Bug Hunt Solution:
# In fake_bookstore.html, the id is 'site-title' (with a hyphen, not an underscore).
# When soup.find() fails to find a tag, it returns None. Calling .text on None raises AttributeError.
header = soup.find("h1", id="site-title")

print("Header text:", header.text.strip())

# Self-check assertion
assert header is not None, "header should not be None!"
assert header.text.strip() == "Welcome to the Fake Bookstore"
print("✅ Test passed!")

Header text: Welcome to the Fake Bookstore
✅ Test passed!


## Reflection

Answer these questions in your own words:

1. Why does BeautifulSoup require `class_="book-title"` with a trailing underscore instead of `class="book-title"`? What error would Python raise if you omitted the underscore?

   **Answer:**
   
   __________________________________________________________________________________________

2. If you search for an element using `soup.find("div", class_="non-existent")`, what value does Python return? Why does calling `.text` on that return value trigger an `AttributeError`?

   **Answer:**
   
   __________________________________________________________________________________________

3. Why is iterating over parent container cards (`<div class="book">`) significantly safer than independently extracting a list of all titles and a list of all prices with two separate `.find_all()` calls?

   **Answer:**
   
   __________________________________________________________________________________________

**Goal:** Explain BeautifulSoup keyword conventions, `NoneType` errors, and the Container Pattern.